## Rule-Based AML Scoring System
Uses the AML Red Flags Database to score customers based on AML indicators.

In [33]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [34]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

Mounted at /content/gdrive


In [35]:
BASE_DIR = '/content/gdrive/MyDrive/AML_Competition'
FEATURE_DIR = Path(BASE_DIR) / 'features'
MODEL_DIR = Path(BASE_DIR) / 'models'
AML_LIBRARY_PATH = BASE_DIR + '/AML_RedFlags_Database.xlsx'

### Load AML Knowledge Library

In [36]:
aml_library = pd.read_excel(AML_LIBRARY_PATH, sheet_name='Master_Indicators')

print(f"   Loaded {len(aml_library)} AML indicators")
print(f"   Categories: {aml_library['Indicator_Category'].nunique()}")
print(f"   High risk indicators: {(aml_library['Risk_Level'] == 'High').sum()}")

   Loaded 151 AML indicators
   Categories: 21
   High risk indicators: 114


### Load Customer Features

In [37]:
df = pd.read_csv(FEATURE_DIR / 'customer_features_enhanced.csv')

print(f"   Loaded {len(df):,} customers")

   Loaded 61,410 customers


### Define Rule-Based Scoring Functions

In [38]:
def score_structuring(row):
    """
    STRUCTURING (ID: STRUCT-*)
    Breaking large transactions into smaller amounts to avoid reporting thresholds
    """
    score = 0.0
    evidence = []

    # Rule 1: Transactions just below $10K threshold
    if 'count_near_10k' in row.index and row['count_near_10k'] >= 5:
        score += 0.40
        evidence.append(f"Structuring: {row['count_near_10k']:.0f} transactions near $10k threshold")

    # Rule 2: Very close to threshold (9800-9999)
    if 'count_very_near_10k' in row.index and row['count_very_near_10k'] >= 3:
        score += 0.30
        evidence.append(f"Structuring: {row['count_very_near_10k']:.0f} transactions $9,800-$9,999")

    # Rule 3: High ratio of round numbers (awareness of reporting)
    if 'ratio_round_100' in row.index and row['ratio_round_100'] > 0.7:
        score += 0.20
        evidence.append(f"Structuring: {row['ratio_round_100']*100:.0f}% round number transactions")

    # Rule 4: Geographic diversity (location hopping)
    if 'unique_cities' in row.index and row['unique_cities'] >= 5:
        score += 0.15
        evidence.append(f"Structuring: Transactions in {row['unique_cities']:.0f} different cities")

    return min(score, 1.0), evidence


def score_profile_mismatch(row):
    """
    PROFILE MISMATCH (ID: PROF-*)
    Activity inconsistent with customer profile
    """
    score = 0.0
    evidence = []

    # Rule 1: Transaction volume exceeds declared income
    if 'spending_to_income_ratio' in row.index:
        if row['spending_to_income_ratio'] > 10:
            score += 0.50
            evidence.append(f"Profile Mismatch: Spending {row['spending_to_income_ratio']:.1f}x declared income")
        elif row['spending_to_income_ratio'] > 5:
            score += 0.30
            evidence.append(f"Profile Mismatch: Spending {row['spending_to_income_ratio']:.1f}x declared income")

    # Rule 2: Business volume doesn't match business size
    if 'volume_to_sales_ratio' in row.index and row['volume_to_sales_ratio'] > 5:
        score += 0.40
        evidence.append(f"Profile Mismatch: Transaction volume {row['volume_to_sales_ratio']:.1f}x declared sales")

    # Rule 3: Young high spender (age < 25, high volume)
    if 'is_young_high_spender' in row.index and row['is_young_high_spender'] == 1:
        score += 0.25
        evidence.append("Profile Mismatch: Young customer with high transaction volume")

    return min(score, 1.0), evidence


def score_layering(row):
    """
    LAYERING (ID: LAYER-*)
    Rapid movement of funds to obscure origin
    """
    score = 0.0
    evidence = []

    # Rule 1: High inflow/outflow ratio (pass-through behavior)
    if 'inflow_outflow_ratio' in row.index:
        if 0.8 < row['inflow_outflow_ratio'] < 1.2:  # Nearly equal in and out
            if 'total_inflow' in row.index and row['total_inflow'] > 100000:
                score += 0.40
                evidence.append(f"Layering: High volume pass-through (${row['total_inflow']:,.0f} in/out)")

    # Rule 2: Rapid turnover (high transactions per day)
    if 'transactions_per_day' in row.index and row['transactions_per_day'] > 5:
        score += 0.30
        evidence.append(f"Layering: {row['transactions_per_day']:.1f} transactions/day")

    # Rule 3: High channel diversity (using many channels)
    if 'channel_diversity' in row.index and row['channel_diversity'] > 0.7:
        score += 0.20
        evidence.append(f"Layering: Using {row['channel_diversity']*7:.0f}/7 different channels")

    return min(score, 1.0), evidence


def score_geographic_risk(row):
    """
    GEOGRAPHIC RISK (ID: GEO-*)
    Transactions with high-risk jurisdictions
    """
    score = 0.0
    evidence = []

    # Rule 1: Drug-producing/transit countries
    if 'drug_country_txn_count' in row.index and row['drug_country_txn_count'] > 0:
        score += 0.45
        evidence.append(f"Geographic Risk: {row['drug_country_txn_count']:.0f} transactions to drug-producing countries")

    # Rule 2: FATF high-risk countries
    if 'high_risk_fatf_txn_count' in row.index and row['high_risk_fatf_txn_count'] > 0:
        score += 0.50
        evidence.append(f"Geographic Risk: {row['high_risk_fatf_txn_count']:.0f} transactions to FATF blacklist countries")

    # Rule 3: Offshore tax havens
    if 'offshore_center_txn_count' in row.index and row['offshore_center_txn_count'] > 0:
        score += 0.35
        evidence.append(f"Geographic Risk: {row['offshore_center_txn_count']:.0f} transactions to offshore centers")

    # Rule 4: High international ratio
    if 'international_ratio' in row.index and row['international_ratio'] > 0.5:
        score += 0.20
        evidence.append(f"Geographic Risk: {row['international_ratio']*100:.0f}% international transactions")

    return min(score, 1.0), evidence


def score_account_activity(row):
    """
    ACCOUNT ACTIVITY (ID: ACCT-*)
    Unusual account behavior
    """
    score = 0.0
    evidence = []

    # Rule 1: New customer with immediate high activity
    if 'customer_tenure_days' in row.index and 'transaction_count_total' in row.index:
        if row['customer_tenure_days'] < 90 and row['transaction_count_total'] > 20:
            score += 0.35
            evidence.append(f"Account Activity: New account ({row['customer_tenure_days']:.0f} days) with {row['transaction_count_total']:.0f} transactions")

    # Rule 2: High behavioral variance (unstable patterns)
    if 'monthly_txn_cv' in row.index and row['monthly_txn_cv'] > 1.5:
        score += 0.25
        evidence.append("Account Activity: Highly unstable transaction patterns")

    # Rule 3: Night transactions (unusual timing)
    if 'night_transaction_ratio' in row.index and row['night_transaction_ratio'] > 0.3:
        score += 0.20
        evidence.append(f"Account Activity: {row['night_transaction_ratio']*100:.0f}% transactions at night")

    return min(score, 1.0), evidence


def score_cash_intensive(row):
    """
    CASH INTENSIVE (Related to multiple indicator categories)
    Heavy use of cash (typical in money laundering)
    """
    score = 0.0
    evidence = []

    # Rule 1: High cash ratio
    if 'cash_ratio' in row.index and row['cash_ratio'] > 0.6:
        score += 0.40
        evidence.append(f"Cash Intensive: {row['cash_ratio']*100:.0f}% cash transactions")

    # Rule 2: High ATM usage
    if 'count_abm' in row.index and 'transaction_count_total' in row.index:
        abm_ratio = row['count_abm'] / (row['transaction_count_total'] + 1)
        if abm_ratio > 0.5:
            score += 0.25
            evidence.append(f"Cash Intensive: {abm_ratio*100:.0f}% ATM transactions")

    return min(score, 1.0), evidence


def score_wire_transfers(row):
    """
    WIRE TRANSFERS (ID: WIRE-*)
    Suspicious wire transfer patterns
    """
    score = 0.0
    evidence = []

    # Rule 1: High wire transfer volume
    if 'count_wire' in row.index and row['count_wire'] > 10:
        score += 0.30
        evidence.append(f"Wire Transfers: {row['count_wire']:.0f} wire transfers")

    # Rule 2: International wires
    if 'international_txn_count' in row.index and row['international_txn_count'] > 5:
        score += 0.25
        evidence.append(f"Wire Transfers: {row['international_txn_count']:.0f} international transfers")

    return min(score, 1.0), evidence

### Apply Rules to Customers

In [39]:
rule_results = []

for idx, row in df.iterrows():
    # Apply all scoring functions
    struct_score, struct_evidence = score_structuring(row)
    profile_score, profile_evidence = score_profile_mismatch(row)
    layer_score, layer_evidence = score_layering(row)
    geo_score, geo_evidence = score_geographic_risk(row)
    acct_score, acct_evidence = score_account_activity(row)
    cash_score, cash_evidence = score_cash_intensive(row)
    wire_score, wire_evidence = score_wire_transfers(row)

    # Get pre-computed AML composite score if available
    aml_composite = row.get('aml_risk_composite', 0)

    # Weighted composite score
    composite_score = (
        struct_score * 0.10 +
        profile_score * 0.10 +
        layer_score * 0.05 +
        geo_score * 0.10 +
        acct_score * 0.05 +
        cash_score * 0.02 +
        wire_score * 0.03 +
        aml_composite * 0.55
    )

    # Collect all evidence
    all_evidence = (
        struct_evidence + profile_evidence + layer_evidence +
        geo_evidence + acct_evidence + cash_evidence + wire_evidence
    )

    # Count triggered rules
    rule_count = sum([
        struct_score > 0, profile_score > 0, layer_score > 0,
        geo_score > 0, acct_score > 0, cash_score > 0, wire_score > 0
    ])

    rule_results.append({
        'customer_id': row['customer_id'],
        'rule_based_score': composite_score,
        'structuring_score': struct_score,
        'profile_mismatch_score': profile_score,
        'layering_score': layer_score,
        'geographic_risk_score': geo_score,
        'account_activity_score': acct_score,
        'cash_intensive_score': cash_score,
        'wire_transfer_score': wire_score,
        'rules_triggered': rule_count,
        'evidence': ' | '.join(all_evidence) if all_evidence else 'No red flags detected'
    })

    # Progress indicator
    if (idx + 1) % 10000 == 0:
        print(f"   Processed {idx+1:,} customers...")

rule_scores_df = pd.DataFrame(rule_results)

print(f"   Scored {len(rule_scores_df):,} customers")

   Processed 10,000 customers...
   Processed 20,000 customers...
   Processed 30,000 customers...
   Processed 40,000 customers...
   Processed 50,000 customers...
   Processed 60,000 customers...
   Scored 61,410 customers


### Analyze Rule-Based Scores

In [40]:
print(f"\n Rule-Based Score Distribution:")
print(f"   Min:  {rule_scores_df['rule_based_score'].min():.3f}")
print(f"   25%:  {rule_scores_df['rule_based_score'].quantile(0.25):.3f}")
print(f"   50%:  {rule_scores_df['rule_based_score'].quantile(0.50):.3f}")
print(f"   75%:  {rule_scores_df['rule_based_score'].quantile(0.75):.3f}")
print(f"   Max:  {rule_scores_df['rule_based_score'].max():.3f}")
print(f"   Mean: {rule_scores_df['rule_based_score'].mean():.3f}")

print(f"\n Rules Triggered Distribution:")
print(rule_scores_df['rules_triggered'].value_counts().sort_index())

# Top scored customers
print(f"\n Top 10 Highest Rule-Based Scores:")
top_10 = rule_scores_df.nlargest(10, 'rule_based_score')
print(top_10[['customer_id', 'rule_based_score', 'rules_triggered']])

# Merge with labels if available
if 'label' in df.columns:
    rule_scores_df = rule_scores_df.merge(
        df[['customer_id', 'label']],
        on='customer_id',
        how='left'
    )

    labeled_mask = rule_scores_df['label'].notna()
    if labeled_mask.sum() > 0:
        clean = rule_scores_df[rule_scores_df['label'] == 0]
        susp = rule_scores_df[rule_scores_df['label'] == 1]

        print(f"\n Rule Score by Actual Label:")
        print(f"   Clean customers: Mean = {clean['rule_based_score'].mean():.3f}")
        print(f"   Suspicious customers: Mean = {susp['rule_based_score'].mean():.3f}")
        print(f"   Difference: {susp['rule_based_score'].mean() - clean['rule_based_score'].mean():.3f}")

        if susp['rule_based_score'].mean() > clean['rule_based_score'].mean():
            print(f"   Rules successfully separate suspicious from clean!")
        else:
            print(f"   Rules need adjustment")


 Rule-Based Score Distribution:
   Min:  0.000
   25%:  0.196
   50%:  0.217
   75%:  0.246
   Max:  0.538
   Mean: 0.232

 Rules Triggered Distribution:
rules_triggered
0      909
1     4611
2    27908
3    20707
4     6643
5      632
Name: count, dtype: int64

 Top 10 Highest Rule-Based Scores:
           customer_id  rule_based_score  rules_triggered
59448  SYNID0200763817          0.538220                5
60223  SYNID0200855360          0.530744                5
55092  SYNID0200242911          0.523019                5
58032  SYNID0200599266          0.521517                5
55324  SYNID0200272812          0.519852                5
56317  SYNID0200394255          0.519852                5
54334  SYNID0200151421          0.518220                5
57086  SYNID0200485667          0.518220                5
56082  SYNID0200364216          0.517809                5
61404  SYNID0200999280          0.512809                5

 Rule Score by Actual Label:
   Clean customers: Mean = 0.208


### Save Rule-Based Scores

In [41]:
output_path = MODEL_DIR / 'rule_based_scores.csv'
rule_scores_df.to_csv(output_path, index=False)

print(f"   Saved to: {output_path}")


   Saved to: /content/gdrive/MyDrive/AML_Competition/models/rule_based_scores.csv


### Summary

In [42]:
print(f"\n Summary:")
print(f"   Customers scored: {len(rule_scores_df):,}")
print(f"   High-risk (score > 0.5): {(rule_scores_df['rule_based_score'] > 0.5).sum():,}")
print(f"   Medium-risk (score 0.3-0.5): {((rule_scores_df['rule_based_score'] >= 0.3) & (rule_scores_df['rule_based_score'] <= 0.5)).sum():,}")

print(f"\n Rule Categories:")
print(f"   Structuring: {(rule_scores_df['structuring_score'] > 0).sum():,} customers flagged")
print(f"   Profile Mismatch: {(rule_scores_df['profile_mismatch_score'] > 0).sum():,} customers flagged")
print(f"   Layering: {(rule_scores_df['layering_score'] > 0).sum():,} customers flagged")
print(f"   Geographic Risk: {(rule_scores_df['geographic_risk_score'] > 0).sum():,} customers flagged")
print(f"   Account Activity: {(rule_scores_df['account_activity_score'] > 0).sum():,} customers flagged")


 Summary:
   Customers scored: 61,410
   High-risk (score > 0.5): 21
   Medium-risk (score 0.3-0.5): 9,399

 Rule Categories:
   Structuring: 39,411 customers flagged
   Profile Mismatch: 10,405 customers flagged
   Layering: 14,239 customers flagged
   Geographic Risk: 33,066 customers flagged
   Account Activity: 272 customers flagged
